# DEM-Based Satellite Line-of-Sight (LOS) Calculator — GPU Accelerated

This notebook computes whether every pixel in a Digital Elevation Model (DEM) has an
unobstructed line-of-sight to a satellite at a given epoch.  
Calculations run on the **GPU** using [CuPy](https://cupy.dev/) for maximum throughput.

**Runtime requirement:** In Google Colab go to *Runtime → Change runtime type* and select **T4 GPU**.

---
**Workflow**
1. Install dependencies
2. Upload / mount a GeoTIFF DEM file
3. Configure satellite TLE + observation epoch
4. Run GPU LOS kernel
5. Visualise and export results


## 0 · Check GPU

In [ ]:
import subprocess
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "⚠️  No GPU detected. Switch to a GPU runtime in Colab.")

## 1 · Install Dependencies

In [ ]:
# Detect CUDA version and install the matching CuPy wheel
import subprocess, re

nvcc = subprocess.run(["nvcc", "--version"], capture_output=True, text=True)
match = re.search(r"release (\d+\.\d+)", nvcc.stdout)
cuda_ver = match.group(1).replace(".", "") if match else "12x"
major = int(cuda_ver[:2]) if len(cuda_ver) >= 2 else 12
cupy_pkg = f"cupy-cuda{major}x"

print(f"Installing {cupy_pkg} ...")
!pip install -q rasterio pyproj sgp4 tqdm matplotlib {cupy_pkg}

## 2 · Imports

In [ ]:
import numpy as np
import rasterio
from rasterio.crs import CRS
from pyproj import Transformer
from sgp4.api import Satrec, jday
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from tqdm.notebook import tqdm
from datetime import datetime, timezone
import os

# Use GPU if available, fall back to CPU
try:
    import cupy as cp
    xp = cp
    print(f"✅ GPU backend: CuPy {cp.__version__} — device: {cp.cuda.Device().use()}")
except ImportError:
    xp = np
    print("⚠️  CuPy not available — falling back to NumPy (CPU).")

## 3 · Configuration

Edit the variables in this cell to match your DEM file and target satellite.

In [ ]:
# ── DEM file ─────────────────────────────────────────────────────────────────
# Upload your GeoTIFF via the Colab file browser, or mount Google Drive
DEM_PATH = "/content/dem.tif"          # ← change to your uploaded file path

# ── Satellite TLE ────────────────────────────────────────────────────────────
# Paste a current TLE from https://celestrak.org/
TLE_LINE1 = "1 25544U 98067A   24001.50000000  .00002182  00000+0  40768-4 0  9994"
TLE_LINE2 = "2 25544  51.6461 339.1840 0003086  27.3272  49.2141 15.50167545432753"

# ── Observation epoch (UTC) ──────────────────────────────────────────────────
EPOCH_UTC = datetime(2024, 1, 1, 12, 0, 0, tzinfo=timezone.utc)

# ── Ray-march resolution ─────────────────────────────────────────────────────
# Number of sample points along each LOS ray.
# Higher → more accurate but slower.  256–512 is a good starting point.
NUM_STEPS = 256

# ── Output ───────────────────────────────────────────────────────────────────
OUTPUT_PNG  = "/content/los_map.png"
OUTPUT_TIFF = "/content/los_map.tif"  # set to None to skip GeoTIFF export

## 4 · Upload DEM (Colab only)

Run this cell to upload a DEM file interactively, **or** skip it if you already
mounted Google Drive and set `DEM_PATH` above.

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()          # opens the file-picker dialog
    if uploaded:
        fname = next(iter(uploaded))
        DEM_PATH = f"/content/{fname}"
        print(f"DEM set to: {DEM_PATH}")
except ImportError:
    print("Not running in Colab — using DEM_PATH from Configuration cell.")

## 5 · Load DEM

In [ ]:
with rasterio.open(DEM_PATH) as src:
    dem_np   = src.read(1).astype(np.float32)   # elevation in metres
    transform = src.transform
    crs       = src.crs
    bounds    = src.bounds
    nrows, ncols = dem_np.shape

print(f"DEM loaded: {ncols}×{nrows} pixels")
print(f"CRS: {crs.to_string()}")
print(f"Bounds: {bounds}")
print(f"Elevation range: {dem_np.min():.1f} – {dem_np.max():.1f} m")

# Build lat/lon grids (degrees)
cols_idx = np.arange(ncols)
rows_idx = np.arange(nrows)
col_grid, row_grid = np.meshgrid(cols_idx, rows_idx)

# Convert pixel centres to geographic coordinates
xs, ys = rasterio.transform.xy(transform, row_grid.ravel(), col_grid.ravel())
xs = np.array(xs, dtype=np.float64).reshape(nrows, ncols)
ys = np.array(ys, dtype=np.float64).reshape(nrows, ncols)

# Reproject to WGS84 lat/lon if necessary
if not crs.is_geographic:
    to_wgs84 = Transformer.from_crs(crs, CRS.from_epsg(4326), always_xy=True)
    lon_grid, lat_grid = to_wgs84.transform(xs, ys)
else:
    lon_grid, lat_grid = xs, ys

print("Lat range:",  lat_grid.min(), "–", lat_grid.max())
print("Lon range:",  lon_grid.min(), "–", lon_grid.max())

## 6 · Compute Satellite Position

In [ ]:
def eci_to_ecef(pos_eci_km, epoch_utc: datetime):
    """Rotate ECI position vector to ECEF using GMST."""
    import math
    # Approximate GMST (radians)
    jd, fr = jday(epoch_utc.year, epoch_utc.month, epoch_utc.day,
                  epoch_utc.hour, epoch_utc.minute,
                  epoch_utc.second + epoch_utc.microsecond * 1e-6)
    T = ((jd + fr) - 2451545.0) / 36525.0
    gmst = (67310.54841 + (876600 * 3600 + 8640184.812866) * T
            + 0.093104 * T**2 - 6.2e-6 * T**3) % 86400.0
    gmst_rad = gmst / 240.0 * math.pi / 180.0
    cos_g, sin_g = math.cos(gmst_rad), math.sin(gmst_rad)
    x, y, z = pos_eci_km
    return np.array([cos_g * x + sin_g * y, -sin_g * x + cos_g * y, z])


def ecef_to_lla(ecef_km):
    """Convert ECEF (km) to geodetic lat (deg), lon (deg), alt (km)."""
    x, y, z = ecef_km * 1000.0   # convert to metres
    a  = 6378137.0               # WGS-84 semi-major axis (m)
    f  = 1 / 298.257223563
    b  = a * (1 - f)
    e2 = 1 - (b / a) ** 2
    lon = np.degrees(np.arctan2(y, x))
    p   = np.sqrt(x**2 + y**2)
    lat = np.degrees(np.arctan2(z, p * (1 - e2)))   # initial approximation
    for _ in range(5):                               # Bowring iteration
        sin_lat = np.sin(np.radians(lat))
        N   = a / np.sqrt(1 - e2 * sin_lat**2)
        lat = np.degrees(np.arctan2(z + e2 * N * sin_lat, p))
    sin_lat = np.sin(np.radians(lat))
    N   = a / np.sqrt(1 - e2 * sin_lat**2)
    alt = p / np.cos(np.radians(lat)) - N
    return lat, lon, alt / 1000.0   # alt in km


satellite = Satrec.twoline2rv(TLE_LINE1, TLE_LINE2)
jd, fr = jday(EPOCH_UTC.year, EPOCH_UTC.month, EPOCH_UTC.day,
              EPOCH_UTC.hour, EPOCH_UTC.minute,
              EPOCH_UTC.second + EPOCH_UTC.microsecond * 1e-6)
error, pos_eci, _ = satellite.sgp4(jd, fr)

if error != 0:
    raise RuntimeError(f"SGP4 propagation error code {error}")

pos_ecef = eci_to_ecef(pos_eci, EPOCH_UTC)
sat_lat, sat_lon, sat_alt_km = ecef_to_lla(pos_ecef)

print(f"Satellite position at {EPOCH_UTC.isoformat()}:")
print(f"  Lat:  {sat_lat:.4f}°")
print(f"  Lon:  {sat_lon:.4f}°")
print(f"  Alt:  {sat_alt_km:.1f} km")

SAT_ALT_M = sat_alt_km * 1000.0   # altitude above WGS-84 ellipsoid (m)

## 7 · GPU LOS Kernel

For each DEM pixel the kernel marches a ray from the pixel toward the satellite,
sampling the DEM at `NUM_STEPS` intermediate points and checking whether any
sample elevation exceeds the straight-line ray height at that distance.

In [ ]:
def bilinear_sample(dem, row_f, col_f):
    """Bilinear interpolation — works with both NumPy and CuPy arrays."""
    h, w = dem.shape
    r0 = xp.clip(xp.floor(row_f).astype(xp.int32), 0, h - 2)
    c0 = xp.clip(xp.floor(col_f).astype(xp.int32), 0, w - 2)
    dr = row_f - r0.astype(xp.float32)
    dc = col_f - c0.astype(xp.float32)
    return ((1 - dr) * (1 - dc) * dem[r0,     c0    ] +
            (1 - dr) *      dc  * dem[r0,     c0 + 1] +
                 dr  * (1 - dc) * dem[r0 + 1, c0    ] +
                 dr  *      dc  * dem[r0 + 1, c0 + 1])


def compute_los(dem_np, lat_grid, lon_grid,
                sat_lat_deg, sat_lon_deg, sat_alt_m,
                transform, num_steps=256):
    """
    Returns a boolean grid (same shape as dem_np) where True means
    the pixel has unobstructed LOS to the satellite.
    """
    dem  = xp.asarray(dem_np,  dtype=xp.float32)
    lat  = xp.asarray(lat_grid, dtype=xp.float64)
    lon  = xp.asarray(lon_grid, dtype=xp.float64)
    nrows, ncols = dem.shape

    # Pixel size in degrees (approximate, from the affine transform)
    deg_per_row = abs(transform.e)   # latitude pixel size
    deg_per_col = abs(transform.a)   # longitude pixel size

    # Satellite position in the same coordinate space
    s_lat = xp.float64(sat_lat_deg)
    s_lon = xp.float64(sat_lon_deg)
    s_alt = xp.float32(sat_alt_m)

    # Direction vectors from each pixel toward satellite (in degrees & metres)
    dlat = s_lat - lat                   # (nrows, ncols)
    dlon = s_lon - lon
    dalt = (s_alt - dem).astype(xp.float64)   # satellite is far above terrain

    visible = xp.ones((nrows, ncols), dtype=xp.bool_)

    for step in range(1, num_steps):
        t = xp.float64(step) / xp.float64(num_steps)

        # Interpolated ray position
        ray_lat = lat + t * dlat
        ray_lon = lon + t * dlon
        ray_alt = dem.astype(xp.float64) + t * dalt   # height above ellipsoid

        # Convert geographic position to fractional DEM pixel indices
        # (inverse of rasterio.transform.xy)
        row_f = (xp.float64(lat_grid[0,  0]) - ray_lat)  / xp.float64(deg_per_row)
        col_f = (ray_lon - xp.float64(lon_grid[0, 0]))    / xp.float64(deg_per_col)

        # Mask pixels outside the DEM extent
        in_bounds = ((row_f >= 0) & (row_f < nrows - 1) &
                     (col_f >= 0) & (col_f < ncols - 1))

        # Sample terrain elevation at this ray position
        row_f_clamped = xp.clip(row_f, 0, nrows - 1).astype(xp.float32)
        col_f_clamped = xp.clip(col_f, 0, ncols - 1).astype(xp.float32)
        terrain_h = bilinear_sample(dem, row_f_clamped, col_f_clamped)

        # If terrain elevation exceeds ray height → blocked
        blocked = in_bounds & (terrain_h.astype(xp.float64) > ray_alt)
        visible &= ~blocked

    if xp is not np:
        visible = cp.asnumpy(visible)
    return visible


print("Running GPU LOS computation …")
los_map = compute_los(
    dem_np, lat_grid, lon_grid,
    sat_lat, sat_lon, SAT_ALT_M,
    transform, num_steps=NUM_STEPS
)
print(f"Done. Visible pixels: {los_map.sum():,} / {los_map.size:,} "
      f"({100 * los_map.mean():.1f}%)")

## 8 · Visualise Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- DEM ---
im0 = axes[0].imshow(dem_np, cmap="terrain",
                     extent=[lon_grid.min(), lon_grid.max(),
                             lat_grid.min(), lat_grid.max()],
                     origin="upper", aspect="auto")
plt.colorbar(im0, ax=axes[0], label="Elevation (m)")
axes[0].set_title("DEM — Terrain Elevation")
axes[0].set_xlabel("Longitude (°)")
axes[0].set_ylabel("Latitude (°)")

# --- LOS map ---
cmap_los = mcolors.ListedColormap(["#d32f2f", "#388e3c"])   # red=blocked, green=visible
im1 = axes[1].imshow(los_map.astype(np.uint8), cmap=cmap_los, vmin=0, vmax=1,
                     extent=[lon_grid.min(), lon_grid.max(),
                             lat_grid.min(), lat_grid.max()],
                     origin="upper", aspect="auto")
cbar = plt.colorbar(im1, ax=axes[1], ticks=[0.25, 0.75])
cbar.ax.set_yticklabels(["Blocked", "Visible"])
axes[1].set_title(f"LOS Map — Satellite @ {sat_lat:.2f}°, {sat_lon:.2f}°, {sat_alt_km:.0f} km")
axes[1].set_xlabel("Longitude (°)")
axes[1].set_ylabel("Latitude (°)")

# Mark sub-satellite point if inside DEM extent
if (lon_grid.min() <= sat_lon <= lon_grid.max() and
        lat_grid.min() <= sat_lat <= lat_grid.max()):
    axes[1].plot(sat_lon, sat_lat, "w*", markersize=14, label="Sub-satellite point")
    axes[1].legend(loc="upper right")

plt.tight_layout()
plt.savefig(OUTPUT_PNG, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figure saved → {OUTPUT_PNG}")

## 9 · Export LOS Map as GeoTIFF

In [ ]:
if OUTPUT_TIFF:
    with rasterio.open(DEM_PATH) as src:
        profile = src.profile

    profile.update(dtype=rasterio.uint8, count=1, compress="lzw", nodata=None)
    with rasterio.open(OUTPUT_TIFF, "w", **profile) as dst:
        dst.write(los_map.astype(np.uint8), 1)

    print(f"GeoTIFF saved → {OUTPUT_TIFF}")
    print("  0 = blocked, 1 = visible")
else:
    print("GeoTIFF export skipped (OUTPUT_TIFF is None).")

## 10 · Download Results (Colab only)

In [ ]:
try:
    from google.colab import files
    for path in [OUTPUT_PNG, OUTPUT_TIFF]:
        if path and os.path.exists(path):
            files.download(path)
except ImportError:
    print(f"Results saved to {OUTPUT_PNG} and {OUTPUT_TIFF}")